# 📁 Toxic comment detection, using machine learning

The objective of this project is to build a machine learning model capable of classifying online comments as toxic or non-toxic.

Online platforms face increasing challenges related to abusive, harmful, or inappropriate content. Automating toxic comment detection is critical for:
- Content moderation
- User safety
- Brand protection
- Scalable platform governance

This project demonstrates the full pipeline of a classification problem, from text preprocessing to model evaluation.

_____________________________________________________________________________________

# CONTEXT

Online discussions are an essential part of modern communication, but they can often be disrupted by toxic comments. Abuse, harassment, and disrespectful language discourage people from expressing their opinions and participating in open conversations. As a result, many users abandon discussions or avoid certain platforms altogether. This issue poses a significant challenge for online communities, which sometimes choose to restrict or completely disable comments to maintain a respectful environment. To address this problem, researchers and developers are working on tools to detect and reduce toxic behavior online. One promising solution is the development of automatic text classification models that can identify toxic comments in real-time.

The aim of this case study is to build a machine learning model capable of detecting whether a comment is toxic or non-toxic

# DATA DESCRIPTION

The dataset contains:
- 50,000 users
- Engagement metrics (sessions, pages viewed, time spent)
- Transaction information
- Revenue data


The datasets is composed of 2 columns: 
- comment_text -> contains the textual content of each comment
- toxic -> labeled 1 for toxic comment -> labeled 0 for non toxic comment

# METHODOLGIE

For this case resolution we have to predict if comments are toxic or not, based on our dataset so we will create several machine learning model, to find the best one to deploy the model. For that we will follow the data analytics lifecycle:
1. Data discovery 
2. Data preparation 
3. Model planning
4. Model building 
5. Model Evaluation
6. Results & Recommendations

# 1. Data discovery

In [1]:
#Load the needed packages 
import pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt
import seaborn as sns
from imblearn.over_sampling import SMOTE
import re
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split 
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
import xgboost as xgb
import lightgbm as lgb

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/constancehenin/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
#Load the dataset
df = pd.read_csv("subset_14.csv")
print(df.head(10))

                                        comment_text  toxic
0             No. DO IT THROUGH THE PROPER CHANNELS.      0
1  "\r\nAnd this should override an admin, why? -...      0
2                Once again..Am I care?? hahaha Shah      0
3  , I suggest you review WP:WEIGHT as well as WP...      0
4  "\r\n\r\n Change to Gill Coliseum page \r\nYou...      1
5         [Cup ref blowtorch turns on Craig Joubert]      0
6  Hey, Stevo. \r\n\r\nLooks like I've got your n...      0
7  Bell? \r\n\r\nThe article says the original de...      0
8  This article is ripe with inaccuracies and omi...      0
9                       I don't see why it should be      0


In [3]:
df['comment_text'] = df['comment_text'].astype(str)   #to insure that the type is a string
print(df['toxic'].value_counts())                     #to see if the dataset is balanced 
df.info()

toxic
0    13000
1     7000
Name: count, dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   comment_text  20000 non-null  object
 1   toxic         20000 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 312.6+ KB


- We can see that the dataset is unbalanced we have more non toxic comments than toxic comments, we will need to fix it by rebalancing the dataset


In [4]:
print(df.isna().sum())

comment_text    0
toxic           0
dtype: int64


-  We have no missing value 

# 2. Data Preparation

In [5]:
#Clean the text of the comments
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()                     #put all text in lowercase
    text = re.sub(r"[^\x01-\x7F]", "", text)     #remove emojis 
    text = re.sub(r"[^\w\s]", "", text)          #remove the punctuation
    text = re.sub(r"\d+", "", text)              #remove the numbers
    words = text.split()                         #tokenize
    words = [word for word in words if word not in stop_words]  #remove stopwords
    return " ".join(words)

df['clean_comment_text'] = df['comment_text'].astype(str).apply(clean_text)  #to apply our clean function to the actual comments

In [6]:
df_clean = df[['clean_comment_text', 'toxic']].copy()  #create the new dataframe with the clean comments and the toxic column

In [7]:
print(df_clean.head(10))

                                  clean_comment_text  toxic
0                                    proper channels      0
1                                override admin talk      0
2                           againam care hahaha shah      0
3                 suggest review wpweight well wpspa      0
4  change gill coliseum page fuckin douche bag on...      1
5              cup ref blowtorch turns craig joubert      0
6  hey stevo looks like ive got number pray tell ...      0
7  bell article says original design called large...      0
8      article ripe inaccuracies omissions see title      0
9                                           dont see      0


# 3. Model Planning

In [8]:
#Split the data into features (X) and target variable (y)
#to prepare the dataset for vectorization and modeling
X = df_clean["clean_comment_text"]
y = df_clean["toxic"]

In [9]:
#Vectorization 
#to converts the text into numerical vectors (features) that ML models can understand.
vectorizer = TfidfVectorizer(max_features=30000)
X = vectorizer.fit_transform(X)

In [10]:
#Oversampling
#to adrress the issue of unbalanced dataset see previously, that we have not enough Toxic comment in the database
oversampler = SMOTE(random_state=42)
X_resampled, y_resampled = oversampler.fit_resample(X, y)
print(pd.Series(y_resampled).value_counts())

toxic
0    13000
1    13000
Name: count, dtype: int64


- So now both classes are balanced with 13,000 samples each, so that the dataset is improved for the model performance

In [11]:
#Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)

# 4. Model Building

In this part we will develop several models, in order to be able to choose the best one to implement

In [12]:
#Logistic Regression
lr = LogisticRegression()
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)

print("Logistic Regression")
print(classification_report(y_test, y_pred))

Logistic Regression
              precision    recall  f1-score   support

           0       0.91      0.88      0.90      2589
           1       0.89      0.92      0.90      2611

    accuracy                           0.90      5200
   macro avg       0.90      0.90      0.90      5200
weighted avg       0.90      0.90      0.90      5200



In [13]:
#Random Forest
rfc = RandomForestClassifier()
rfc.fit(X_train, y_train)
y_pred = rfc.predict(X_test)

print("Random Forest")
print(classification_report(y_test, y_pred))

Random Forest
              precision    recall  f1-score   support

           0       0.90      0.93      0.92      2589
           1       0.93      0.90      0.91      2611

    accuracy                           0.91      5200
   macro avg       0.92      0.91      0.91      5200
weighted avg       0.92      0.91      0.91      5200



In [14]:
#Support Vector Machine (SVM)
svm = LinearSVC()
svm.fit(X_train, y_train)
y_pred = svm.predict(X_test)

print("SVM")
print(classification_report(y_test, y_pred))

SVM
              precision    recall  f1-score   support

           0       0.94      0.86      0.90      2589
           1       0.88      0.95      0.91      2611

    accuracy                           0.91      5200
   macro avg       0.91      0.91      0.91      5200
weighted avg       0.91      0.91      0.91      5200



/opt/anaconda3/lib/python3.12/site-packages/sklearn/svm/_classes.py:31: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(


In [15]:
#Gradient Boosting
gbc = GradientBoostingClassifier()
gbc.fit(X_train, y_train)
y_pred = gbc.predict(X_test)

print("Gradient Boosting")
print(classification_report(y_test, y_pred))

Gradient Boosting
              precision    recall  f1-score   support

           0       0.76      0.96      0.85      2589
           1       0.94      0.70      0.80      2611

    accuracy                           0.83      5200
   macro avg       0.85      0.83      0.82      5200
weighted avg       0.85      0.83      0.82      5200



In [16]:
#Naive Bayes
nb = MultinomialNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)

print("Naive Bayes")
print(classification_report(y_test, y_pred))

Naive Bayes
              precision    recall  f1-score   support

           0       0.76      0.96      0.85      2589
           1       0.94      0.70      0.80      2611

    accuracy                           0.83      5200
   macro avg       0.85      0.83      0.82      5200
weighted avg       0.85      0.83      0.82      5200



In [17]:
#XGBoost
xgb = XGBClassifier()
xgb.fit(X_train, y_train)
y_pred = xgb.predict(X_test)

print("XGBoost")
print(classification_report(y_test, y_pred))

XGBoost
              precision    recall  f1-score   support

           0       0.84      0.95      0.89      2589
           1       0.94      0.82      0.88      2611

    accuracy                           0.89      5200
   macro avg       0.89      0.89      0.89      5200
weighted avg       0.89      0.89      0.89      5200



In [18]:
#Decision Tree
dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)
y_pred = dt.predict(X_test)

print("Decision Tree")
print(classification_report(y_test, y_pred))

Decision Tree
              precision    recall  f1-score   support

           0       0.88      0.87      0.87      2589
           1       0.87      0.88      0.88      2611

    accuracy                           0.88      5200
   macro avg       0.88      0.88      0.88      5200
weighted avg       0.88      0.88      0.88      5200



# 5. Model Evaluation

In [19]:
#Create a function to extract the performance of each model 
def print_model_evaluation(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred)
    confusion_mat = confusion_matrix(y_test, y_pred)

    print(f"### {model_name} ###")
    print(report)
    print("Confusion Matrix:")
    print(confusion_mat)
    print("\n")

In [20]:
print_model_evaluation(lr, X_test, y_test, "Logistic Regression")
print_model_evaluation(rfc, X_test, y_test,"Random Forest")
print_model_evaluation(svm, X_test, y_test,"SVM")
print_model_evaluation(gbc, X_test, y_test, "Gradient Boosting")
print_model_evaluation(nb, X_test, y_test, "Naive Bayes")
print_model_evaluation(xgb, X_test, y_test, "XGBoost")
print_model_evaluation(dt, X_test, y_test, "Decision Tree")

### Logistic Regression ###
              precision    recall  f1-score   support

           0       0.91      0.88      0.90      2589
           1       0.89      0.92      0.90      2611

    accuracy                           0.90      5200
   macro avg       0.90      0.90      0.90      5200
weighted avg       0.90      0.90      0.90      5200

Confusion Matrix:
[[2283  306]
 [ 215 2396]]


### Random Forest ###
              precision    recall  f1-score   support

           0       0.90      0.93      0.92      2589
           1       0.93      0.90      0.91      2611

    accuracy                           0.91      5200
   macro avg       0.92      0.91      0.91      5200
weighted avg       0.92      0.91      0.91      5200

Confusion Matrix:
[[2411  178]
 [ 266 2345]]


### SVM ###
              precision    recall  f1-score   support

           0       0.94      0.86      0.90      2589
           1       0.88      0.95      0.91      2611

    accuracy              

# 6. Results & Recommendations

So, after running our different models (7 models) and evaluating their performance, we can see that all our models are quite good, as they all have an accuracy of over 80%.

By looking at the performance evaluation of every model, espacially at the confusion matrix, we can see that:

* The SVM model and the Random forest model have the best accuracy of all models; 91%. So, in order to decide between this 2 model wich one is the best, we will look at the False prediction, positive and negative one. 
* We can see that the SVM model has 487 false prediction while the Random Forest model has only 444 false prediction 

So, we can conclude that the Random Forest model is the best one, with the best accuracy and the less false predictions.

Researchers and developers have now a good model, an automatic text classification to detect toxic behavior online.
In order to reduce toxic behavior online they can now operationalize this model by deploying it in order to create the perfect tool to detect toxic comment in real-time. They can for example create an interface like an API or application to use the model and determine if comments are toxic or not. 

Moreover, it could be interesting to retraine the model with more data (if they have more available), because by retraining the model the model with more data it could improve its accuracy, and the model could becomme even better.  